In [1]:
# first prompt for getting values
# second prompt for generating of code

In [16]:
import sys
sys.path.append("../src")
import pandas as pd
from utils.utils import generate, eval_generic, to_csv, Output
from utils.models import get_models
from prompts import pal
import json
import subprocess


In [3]:
models = get_models()
df = pd.read_excel("../data/ai_lab_1.xlsx")
gr_truth = pd.read_csv("../data/ground_truth.csv")

In [6]:
def make_preamble(json_data: list[dict]) -> str:
    """Deterministic header injected before model code — not trusted to the model."""
    json_str = json.dumps(json_data)
    return f"import json\ndata = json.loads({repr(json_str)})\n"

In [7]:
ALLOWED_UNITS = {"m2", "sqm", "square meters", "sqft", "sq ft", "ft2", "ha", "ac"}

def validate(raw_json: str):
    """Parse and structurally validate the extracted JSON."""
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError as e:
        print("JSON parse error:", e)
        return None, "json_parse_error"

    if not isinstance(data, list) or len(data) == 0:
        print("Validation error: expected non-empty list, got:", type(data))
        return None, "invalid_structure"

    required_keys = {"property_id", "value", "unit"}
    filtered = []
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            print(f"Validation error: item {i} is not a dict, skipping")
            continue
        missing = required_keys - item.keys()
        if missing:
            print(f"Validation error: item {i} missing keys {missing}, skipping")
            continue
        if item["unit"].lower() not in ALLOWED_UNITS:
            print(f"Skipping unsupported unit: {item['unit']!r} in item {i}")
            continue
        filtered.append(item)

    if not filtered:
        print("Validation error: no valid area items after filtering")
        return None, "no_valid_items"

    return filtered, None

In [8]:
def extract_code(text: str) -> str:
    text = text.strip()
    if "```" in text:
        lines = text.split("\n")
        code_lines = []
        inside_block = False
        for line in lines:
            if line.strip().startswith("```"):
                inside_block = not inside_block
                continue
            if inside_block:
                code_lines.append(line)
        return "\n".join(code_lines).strip()
    return text


In [9]:
def exec_code(generated_code: str, preamble: str):
    logic = extract_code(generated_code)
    full_code = preamble + "\n" + logic
    try:
        result = subprocess.run(
            ["python", "-c", full_code],
            capture_output=True,
            text=True,
            timeout=10
        )
        return result
    except Exception as e:
        print("Execution error:", e)
        return None

In [10]:
def parse_stdout(stdout: str) -> dict | None:
    try:
        return json.loads(stdout.strip())
    except json.JSONDecodeError as e:
        print("stdout parse error:", e, "| raw:", repr(stdout))
        return None

In [ ]:
def generate_pal(model, pid, prompt):
    extract_prompt = pal.make_extract_prompt(pid, prompt)
    raw_json = generate(model, extract_prompt)
    json_data, err = validate(raw_json)

    if err is not None:
        return None

    code_prompt = pal.make_code_gen_prompt()
    generated_code = generate(model, code_prompt)

    preamble = make_preamble(json_data)
    res = exec_code(generated_code, preamble)

    if res is None:
        print("Code execution returned None")
        return None

    if res.returncode == 0:
        print(res.stdout)
        return res.stdout
    else:
        print("Code execution error:", res.stderr)
        return None


In [13]:
for model in models:
    print(f"start for {model.short_name}")
    res = eval_generic(model=model.model_name, df=df, gr_truth=gr_truth, generate_fn=generate_pal)
    to_csv(model.short_name, "pal_prompt.csv", res)
    print(f"done for {model.short_name}")

print("done")

start for llama3.2


  0%|          | 0/117 [00:00<?, ?it/s]

  1%|          | 1/117 [00:08<16:49,  8.70s/it]

Code execution error: Traceback (most recent call last):
  File "<string>", line 14, in <module>
NameError: name 'result' is not defined



  2%|▏         | 2/117 [00:13<12:14,  6.39s/it]

...

{"115": 43801.5478}



 99%|█████████▉| 116/117 [20:09<00:11, 11.72s/it]

{"116": 42300}



100%|██████████| 117/117 [20:21<00:00, 10.44s/it]

{"117": 24000.0}

done for qwen2.5
done


In [14]:
Output.model_validate_json("{\"1\": 51400}").root

{'1': 51400.0}